# Train v21 on Workstation

This notebook extracts the packaged code locally and launches v21 training in-process so logs appear directly in the notebook. Keep API keys in environment variables or a local `.env`; do not paste secrets into cells.

In [ ]:
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

VERSION = 'v21'
DRIVE_CODE_DIR = Path('G:/My Drive/quant-research-workbench/workstation_code') / VERSION
PACKAGE_ZIP = DRIVE_CODE_DIR / f'inhouse_transformer_{VERSION}_workstation.zip'
MANIFEST_PATH = DRIVE_CODE_DIR / 'workstation_manifest.json'
LOCAL_CODE_ROOT = Path('D:/TradingCodes/quant-research-workbench-v21-runtime')

assert PACKAGE_ZIP.exists(), f'Missing package: {PACKAGE_ZIP}'
assert MANIFEST_PATH.exists(), f'Missing manifest: {MANIFEST_PATH}'
manifest = json.loads(MANIFEST_PATH.read_text())
print(json.dumps(manifest, indent=2))

if LOCAL_CODE_ROOT.exists():
    shutil.rmtree(LOCAL_CODE_ROOT)
LOCAL_CODE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as package:
    package.extractall(LOCAL_CODE_ROOT)
sys.path.insert(0, str(LOCAL_CODE_ROOT))
print('installed code at', LOCAL_CODE_ROOT)


In [ ]:
# Edit FLATFILES_ROOT if you copy data from the HDD/Drive path to local SSD/NVMe.
FLATFILES_ROOT = Path(manifest['default_flatfiles_root'])
CACHE_ROOT = Path(manifest['default_cache_root'])
print('flatfiles root:', FLATFILES_ROOT, 'exists=', FLATFILES_ROOT.exists())
print('cache root:', CACHE_ROOT)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
%pip install -q polars pyarrow wandb torchinfo torchview graphviz


In [ ]:
import runpy
import sys

BATCH_SIZE = int(manifest.get('default_batch_size', 4096))
EPOCHS = int(manifest.get('default_epochs', 3))
NUM_WORKERS = int(manifest.get('default_num_workers', 8))
PREFETCH_FACTOR = int(manifest.get('default_prefetch_factor', 4))
MAX_STEPS = 0
COUNT_COVERAGE = False
DRY_RUN = False
REBUILD_CACHE = False

train_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v21' / 'train.py'
args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--train-start-date', manifest['train_start_date'],
    '--train-end-date', manifest['train_end_date'],
    '--validation-start-date', manifest['validation_start_date'],
    '--validation-end-date', manifest['validation_end_date'],
    '--test-start-date', manifest['test_start_date'],
    '--test-end-date', manifest['test_end_date'],
    '--device', 'cuda',
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--prefetch-factor', str(PREFETCH_FACTOR),
    '--tickers', manifest.get('tickers', 'ALL'),
    '--checkpoint-policy', 'last_only',
    '--wandb-entity', manifest['wandb_entity'],
    '--wandb-project', manifest['wandb_project'],
    '--wandb-run-name', manifest['wandb_run_name'],
    '--output-name', manifest['wandb_run_name'],
]
if REBUILD_CACHE:
    args.append('--rebuild-cache')
if COUNT_COVERAGE:
    args.append('--count-coverage')
if DRY_RUN:
    args.append('--dry-run')

print('Running:', ' '.join([str(train_py), *args]))
old_argv = sys.argv[:]
try:
    sys.argv = [str(train_py), *args]
    runpy.run_path(str(train_py), run_name='__main__')
finally:
    sys.argv = old_argv
